<a href="https://colab.research.google.com/github/zephyrroche/FIFA-World-Cup-Statistical-Analysis-and-2026-Score-Prediction/blob/main/FIFA_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import accuracy_score, mean_absolute_error
import joblib

In [2]:
from google.colab import files
data_to_load = files.upload()

Saving clean_fifa_worldcup_matches.csv to clean_fifa_worldcup_matches.csv


In [3]:
df = pd.read_csv("clean_fifa_worldcup_matches.csv")

In [4]:
print(df.head())

    HomeTeam AwayTeam  Year  HomeGoals  AwayGoals  TotalGoals
0     France   Mexico  1930          4          1           5
1  Argentina   France  1930          1          0           1
2      Chile   Mexico  1930          3          0           3
3      Chile   France  1930          1          0           1
4  Argentina   Mexico  1930          6          3           9


In [5]:
required_columns = {'HomeTeam', 'AwayTeam', 'Year', 'HomeGoals', 'AwayGoals', 'TotalGoals'}
if not required_columns.issubset(df.columns):
    raise ValueError("Dataset is missing required columns!")

In [6]:
all_teams = pd.concat([df['HomeTeam'], df['AwayTeam']]).unique()

In [7]:
team_encoder = LabelEncoder()
team_encoder.fit(all_teams)

LabelEncoder()

In [8]:
df['HomeTeamEncoded'] = team_encoder.transform(df['HomeTeam'])
df['AwayTeamEncoded'] = team_encoder.transform(df['AwayTeam'])

In [9]:
X = df[['HomeTeamEncoded', 'AwayTeamEncoded', 'Year']]
y_home = df['HomeGoals']
y_away = df['AwayGoals']

In [10]:
X_train, X_test, y_train_home, y_test_home = train_test_split(X, y_home, test_size=0.2, random_state=42)
X_train, X_test, y_train_away, y_test_away = train_test_split(X, y_away, test_size=0.2, random_state=42)

home_model = RandomForestRegressor(n_estimators=100, random_state=42)
away_model = RandomForestRegressor(n_estimators=100, random_state=42)

home_model.fit(X_train, y_train_home)
away_model.fit(X_train, y_train_away)

RandomForestRegressor(random_state=42)

In [11]:
joblib.dump(home_model, "home_goal_model.pkl")
joblib.dump(away_model, "away_goal_model.pkl")
joblib.dump(team_encoder, "team_encoder.pkl")
print("Models trained and saved successfully!")

Models trained and saved successfully!


In [12]:
from google.colab import files
data_to_load = files.upload()

Saving clean_fifa_worldcup_fixture.csv to clean_fifa_worldcup_fixture.csv


In [13]:
#time for prediction
fixtures_2026 = pd.read_csv("clean_fifa_worldcup_fixture.csv")

In [17]:
fixtures_2026.head()

,home,score,away,year
0,USA,Match 1,Mexico,2026
1,Canada,Match 2,Costa Rica,2026
2,USA,Match 18,Canada,2026
3,Mexico,Match 19,Costa Rica,2026
4,Costa Rica,Match 35,Canada,2026


In [14]:
def encode_team(team):
    if team in team_encoder.classes_:
        return team_encoder.transform([team])[0]
    else:
        return -1

In [20]:
# Ensure proper renaming with correct case
fixtures_2026.rename(columns={'Home': 'HomeTeam', 'Away': 'AwayTeam'}, inplace=True)

# Print column names to verify renaming worked
print("Columns after renaming:", list(fixtures_2026.columns))

# Encode teams
fixtures_2026['HomeTeamEncoded'] = fixtures_2026['home'].apply(encode_team)
fixtures_2026['AwayTeamEncoded'] = fixtures_2026['away'].apply(encode_team)

Columns after renaming: ['home', 'score', 'away', 'year']


In [21]:
fixtures_2026.head()

,home,score,away,year,HomeTeamEncoded,AwayTeamEncoded
0,-1,Match 1,44,2026,-1,-1
1,11,Match 2,15,2026,-1,-1
2,-1,Match 18,11,2026,-1,-1
3,44,Match 19,15,2026,-1,-1
4,15,Match 35,11,2026,-1,-1


In [37]:
X_2026['Year'] = 2026

In [38]:
X_2026 = X_2026[['HomeTeamEncoded', 'AwayTeamEncoded', 'Year']]

In [39]:
fixtures_2026['HomeTeamEncoded'] = fixtures_2026['home'].apply(encode_team)
fixtures_2026['AwayTeamEncoded'] = fixtures_2026['away'].apply(encode_team)

# Drop matches where either team encoding is -1 (i.e., new unseen teams)
fixtures_2026 = fixtures_2026[
    (fixtures_2026['HomeTeamEncoded'] != -1) & (fixtures_2026['AwayTeamEncoded'] != -1)
].copy()  # Ensure we create a new copy

# Prepare dataset for prediction
X_2026 = fixtures_2026[['HomeTeamEncoded', 'AwayTeamEncoded']].copy()  # Create a copy
X_2026.loc['year'] = 2026  # Modify safely to avoid SettingWithCopyWarning

# Load trained models
home_model = joblib.load("home_goal_model.pkl")
away_model = joblib.load("away_goal_model.pkl")

In [40]:
print(X_train.columns)
print(X_2026.columns)

Index(['HomeTeamEncoded', 'AwayTeamEncoded', 'Year'], dtype='object')
Index(['HomeTeamEncoded', 'AwayTeamEncoded'], dtype='object')


In [44]:
# Add tournament year
X_2026['Year'] = 2026

# Match training feature order
X_2026 = X_2026[['HomeTeamEncoded', 'AwayTeamEncoded', 'Year']]

# Predict goals
home_preds = home_model.predict(X_2026)
away_preds = away_model.predict(X_2026)

# Round predictions
fixtures_2026['PredictedHomeGoals'] = np.round(home_preds).astype(int)
fixtures_2026['PredictedAwayGoals'] = np.round(away_preds).astype(int)

# Prevent negatives
fixtures_2026['PredictedHomeGoals'] = fixtures_2026['PredictedHomeGoals'].clip(lower=0)
fixtures_2026['PredictedAwayGoals'] = fixtures_2026['PredictedAwayGoals'].clip(lower=0)

# Create score column
fixtures_2026['Score'] = (
    fixtures_2026['PredictedHomeGoals'].astype(str)
    + " - " +
    fixtures_2026['PredictedAwayGoals'].astype(str)
)

In [50]:
fixtures_2026.head()

,home,score,away,year,HomeTeamEncoded,AwayTeamEncoded,PredictedHomeGoals,PredictedAwayGoals,Score
0,NaN,NaN,NaN,NaN,NaN,NaN,7,1,7 - 1


In [49]:
fixtures_2026[['home', 'score', 'away', 'year']].to_csv("fifa_worldcup_2026_Score_predictions.csv", index=False)

print("Predictions for World Cup 2026 saved successfully!")

Predictions for World Cup 2026 saved successfully!
